# MeguvizCAC v0.6.0-HyperLoop 全量训练（Bundle v3 · Split-Head Hybrid + Nano-Draft L2D-FP + LB-DSM + 0KB Surgery）

完整落地 `docs/roadmap-v0.6.0-self-evolution.md` 与 RTX 4060 Ti 实证突破：
1. **Tier-0 极速门控（0.155 MB `NanoDraftL2DFP`）**：UTF-8 字节哈希 + 深度可分离卷积 + 32-d PIFE，严格验证集零漏检（FN <= 0.25%）早退策略。
2. **4L `Split-Head Hybrid` 主干（零损耗 `MOHAWK` 继承 `v0.5.0-R3` 权重）**：4 路精确 `SWA-64` Softmax 头（短评 `L<=64` 100% 精确注意力） + 4 路双向门控线性 `Bi-DeltaNet-2` 头（支持 `max_len=256`） + `BitDistill SubLN` + 平稳 3 级可学习 `Lifting Haar DWT` 节律残差（零 FFN 带宽偷窃）。
3. **原生检测理论武装**：`Bi-DeltaNet` 4-d 前后向不对称波统计 + 句内 `MIL` 峰值/Span 头 + 并行残差 `TopK-SAE` (`TinyScope-CBM`) + Clean-Whitelist `Two-Way pAUC`。
4. **零代价后处理与冻结几何批评家**：Stage-4a **0 KB 混淆神经元权重折叠手术**（验证集自动门控） + Stage-4b **22 KB 冻结 `LB-DSM` 双流形 Hutchinson 曲率与 OOD 批评家**。
5. **端到端级联共形校准**：`n=2,400` 分层验证集 + 锐利温度烘焙（`T ∈ [0.9, 1.8]`）+ 分长度桶 Smooth PAVA + **Mandap Construction-B 全路径级联 Conformal**。


In [ ]:
# Cell 1: 挂载 Drive、拉取 v0.6.0-HyperLoop 代码、准备语料资产（若已存在自动秒过）
import hashlib, json, os, shutil, subprocess, time, torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
DRIVE_OUT = Path('/content/drive/MyDrive/MeguvizCAC/v060_hyperloop')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / '_session_check.txt').write_text(f'mounted_utc={time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n', encoding='utf-8')

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU！请先连接 GPU 运行时后再执行本单元。')
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB | bf16={torch.cuda.is_bf16_supported()}')

REPO_DIR = '/content/MeguvizCAC'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} -c core.fileMode=false fetch origin && git -C {REPO_DIR} -c core.fileMode=false pull --ff-only
%cd /content/MeguvizCAC
!git log -3 --oneline
!pip -q install -U tokenizers onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub

DATA_DIR = Path('/content/data')
for d in (DATA_DIR / 'stage2', DATA_DIR / 'co-teacher-gate'):
    d.mkdir(parents=True, exist_ok=True)
REL = 'https://github.com/jichuo1/MeguvizCAC/releases/download'

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()

def ensure_dl(url: str, dst: Path, pref: str | None = None):
    if dst.is_file() and dst.stat().st_size > 0 and (pref is None or sha256_file(dst).startswith(pref)):
        print(f'[cache hit] {dst.name}')
        return
    print(f'[download] {dst.name} ...')
    subprocess.run(['curl', '-fsSL', url, '-o', str(dst)], check=True)
    if pref:
        assert sha256_file(dst).startswith(pref), f'SHA mismatch: {dst.name}'

ensure_dl(f'{REL}/v0.2-data/stage2_distill_bundle.zip', DATA_DIR / 'stage2_distill_bundle.zip')
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2
ensure_dl(f'{REL}/v0.2-qwen-raw/stage2_qwen_bundle.zip', DATA_DIR / 'stage2_qwen_bundle.zip')
!unzip -qoj /content/data/stage2_qwen_bundle.zip -d /content/data/stage2
for name, dst, pref in [
    ('v04-qwen38-twins.jsonl', DATA_DIR / 'v04-qwen38-twins.jsonl', '7ff4227c'),
    ('v04-inkling-wk-pairs.jsonl', DATA_DIR / 'v04-inkling-wk-pairs.jsonl', '0e877a7c'),
    ('v04_semantic_intent_cache.npz', DATA_DIR / 'v04_semantic_intent_cache.npz', '9880e38b'),
    ('v04-merged-1m4.jsonl.gz', DATA_DIR / 'v04-merged-1m4.jsonl.gz', '11b63db0'),
    ('v04-twins-clean-pairs.jsonl', DATA_DIR / 'co-teacher-gate' / 'v04-twins-clean-pairs.jsonl', '3cae814d'),
]:
    ensure_dl(f'{REL}/v0.4-data/{name}', dst, pref)

r3_corpus = DATA_DIR / 'v04-merged-1m4-r3.jsonl.gz'
r3_rep = DATA_DIR / 'r3_persona_labelfix_report.json'
if not (r3_corpus.is_file() and r3_rep.is_file()):
    subprocess.run([
        'python3', '-m', 'meguviz_cac.v04_persona_labelfix',
        '--in', str(DATA_DIR / 'v04-merged-1m4.jsonl.gz'),
        '--out', str(r3_corpus),
        '--report', str(r3_rep),
    ], check=True)
print('V06_STEP1_READY | Drive:', DRIVE_OUT)


In [ ]:
# Cell 2: 启动后台抗断线 v0.6.0-HyperLoop 全量训练（自动检测 v0.5-R3 MOHAWK 权重、--probe-vram、双写 Drive 与 Bundle v3 打包）
import sys, importlib
if '/content/MeguvizCAC' not in sys.path:
    sys.path.insert(0, '/content/MeguvizCAC')
import meguviz_cac.colab_v06_launcher as lv06
importlib.reload(lv06)
lv06.run_v06_in_colab(poll_seconds=90)


In [ ]:
# Cell 3: 断线重连监控 / 手动补下载单元
import json
from pathlib import Path
from google.colab import files

done = Path('/content/v06_done.json')
if done.is_file():
    info = json.loads(done.read_text(encoding='utf-8'))
    print(json.dumps(info, ensure_ascii=False, indent=2))
    files.download(info['full_zip'])
else:
    !tail -n 30 /content/log_v06_driver.txt 2>/dev/null || true
    !tail -n 20 /content/log_v06_full.txt 2>/dev/null || true
